# 03 — Entraînement du modèle
> Transfer learning EfficientNet-B3 + fine-tuning + early stopping.

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import matplotlib.pyplot as plt

from config import NUM_EPOCHS, LR, SEED, MODEL_PATH
from src.train import train, set_seed
from src.model import build_model, count_parameters

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device : {device}")
print(f"CUDA disponible : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")

## 1. Architecture du modèle

In [ ]:
model = build_model(freeze_backbone=True)
params = count_parameters(model)
print(f"Paramètres totaux     : {params['total']:,}")
print(f"Paramètres entraînables : {params['trainable']:,}")
print(f"Backbone gelé : {100*(params['total']-params['trainable'])/params['total']:.1f}% des paramètres")
print("\nArchitecture du classifier (tête custom) :")
print(model.classifier)

## 2. Stratégie d'entraînement

**Phase 1** (époques 1→5) : Seule la tête custom est entraînée. Le backbone reste gelé.

**Phase 2** (époques 6→fin) : Fine-tuning — les 3 derniers blocs d'EfficientNet sont dégelés avec un LR réduit (LR/10).

In [ ]:
print("Stratégie d'entraînement :")
print(f"  Époques max      : {NUM_EPOCHS}")
print(f"  LR initial       : {LR}")
print(f"  LR fine-tuning   : {LR/10:.2e}")
print(f"  Optimiseur       : AdamW (weight_decay=1e-4)")
print(f"  Scheduler        : CosineAnnealingLR")
print(f"  Early stopping   : patience=7 époques sur val_loss")
print(f"  Loss             : CrossEntropyLoss avec class_weights")
print(f"  Dégel backbone   : après 5 époques (3 derniers blocs)")

## 3. Lancement de l'entraînement

⏱ **Durée estimée** : 10–15 min sur CPU | 2–3 min sur GPU

In [ ]:
# Lance l'entraînement complet
history = train(
    num_epochs=NUM_EPOCHS,
    lr=LR,
    unfreeze_after=5,
)
print(f"\n✓ Entraînement terminé — Modèle sauvegardé : {MODEL_PATH}")

## 4. Courbes d'apprentissage

In [ ]:
from src.evaluate import plot_training_curves
plot_training_curves(history, save_path='../models/training_curves.png')

## 5. Résumé

In [ ]:
best_val_acc  = max(history['val_acc'])
best_train_acc = max(history['train_acc'])
best_epoch = history['val_acc'].index(best_val_acc) + 1

print("=" * 40)
print("RÉSUMÉ ENTRAÎNEMENT")
print("=" * 40)
print(f"Meilleure val accuracy : {best_val_acc:.4f} (époque {best_epoch})")
print(f"Train accuracy finale  : {best_train_acc:.4f}")
gap = best_train_acc - best_val_acc
print(f"Écart train/val        : {gap:.4f}", end="  ")
if gap > 0.1:
    print("⚠ Overfitting possible")
elif gap < 0:
    print("→ Val meilleur que train (normal avec dropout)")
else:
    print("✓ Bon équilibre")
print(f"\nModèle sauvegardé : {MODEL_PATH}")